# Scarlet collaborator reproduction

Set the data path in the configuration cell, then choose **Run All Cells**. The notebook runs the complete, predeclared A/B/C reproduction and displays a summary and plots.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Configuration — all run-defining choices are explicit here.
# Normally, only DATA_ROOT needs to be changed.
SCARLET_REPO = Path.cwd().resolve()
DATA_ROOT = Path('/path/to/collaborator/data')
OUTPUT_ROOT = SCARLET_REPO / 'benchmark_artifacts' / 'collaborator_reproduction'

STARTS = ('A', 'B', 'C')
OPTIMIZER = 'variable_projection'
FEATURE = 'centroid_psf'
MAX_ITER = 1500
FIT_DTYPE = 'float64'
CHANNEL_CHUNK_SIZE = 64
OPTIMALITY_TOLERANCE = 1e-4
OPTIMALITY_CHECK_INTERVAL = 20

In [ ]:
run_dirs = {start: OUTPUT_ROOT / f'start{start}' for start in STARTS}
fit_commands = {}
for start, run_dir in run_dirs.items():
    fit_commands[start] = [
        sys.executable, '-m', 'benchmarks.run_collaborator_reproduction',
        '--data-root', str(DATA_ROOT),
        '--output-dir', str(run_dir),
        '--start', start,
        '--max-iter', str(MAX_ITER),
        '--dtype', FIT_DTYPE,
        '--channel-chunk-size', str(CHANNEL_CHUNK_SIZE),
        '--optimizer', OPTIMIZER,
        '--feature', FEATURE,
        '--optimality-tolerance', str(OPTIMALITY_TOLERANCE),
        '--optimality-check-interval', str(OPTIMALITY_CHECK_INTERVAL),
    ]
{start: ' '.join(command) for start, command in fit_commands.items()}

## Run the three declared starts

This is the long-running step. Results are saved under `OUTPUT_ROOT`.

In [ ]:
for start in STARTS:
    print(f'Running declared start {start}', flush=True)
    subprocess.run(fit_commands[start], cwd=SCARLET_REPO, check=True)

In [ ]:
products = {}
metrics_paths = {}
plot_dirs = {}
for start, run_dir in run_dirs.items():
    products[start] = run_dir / f'scarlet_matched_start{start}.npz'
    metrics_paths[start] = run_dir / 'scarlet_matched_metrics.json'
    plot_dirs[start] = run_dir / 'plots_frame_aligned'
    plot_command = [
        sys.executable, '-m', 'benchmarks.plot_collaborator_reproduction',
        '--product', str(products[start]),
        '--metrics', str(metrics_paths[start]),
        '--output-dir', str(plot_dirs[start]),
    ]
    subprocess.run(plot_command, cwd=SCARLET_REPO, check=True)

In [ ]:
reports = {
    start: json.loads(path.read_text())
    for start, path in metrics_paths.items()
}
observable_summary = {
    start: {
        'iterations': report['iterations'],
        'optimality_converged': report['optimality_converged'],
        'projected_gradient': report['parameter_relative_projected_gradient'],
        'chi_square_per_voxel': report['residual']['chi_square_per_voxel'],
        'lag1_autocorrelation': report['residual']['lag1_autocorrelation'],
    }
    for start, report in reports.items()
}
observable_summary

In [ ]:
recovery_summary = {}
for start, report in reports.items():
    recovery_summary[start] = [
        {
            'spectrum_relative_l2': source['spectrum']['relative_l2'],
            'spectrum_24bin_rms': source['spectrum']['binned_fractional_error_rms'],
            'spectrum_24bin_worst': source['spectrum']['binned_fractional_error_max_abs'],
            'morphology_relative_l2': source['morphology']['relative_l2'],
        }
        for source in report['sources']
    ]
recovery_summary  # Injection-only audit; never use this to select a start.

## Plots

Starts A and B should agree. Start C is a known poor local solution and can be identified by its worse chi-square without using the injected truth.

In [ ]:
from IPython.display import Image, Markdown, display

for start in STARTS:
    display(Markdown(f'## Declared start {start}'))
    for name in (
        'scarlet_collaborator_spectra.png',
        'scarlet_collaborator_morphologies.png',
        'scarlet_collaborator_residual.png',
    ):
        display(Image(filename=str(plot_dirs[start] / name)))